In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [3]:
train = pd.read_csv("../data/train.csv")
validation = pd.read_csv("../data/validation.csv")

In [6]:
features = [
    "Vol",
    "lag_15",
    "lag_30",
    "lag_60",
    "lag_120",
    "lag_1day",
    "hour",
    "day_of_week"
]

target = "target_15min"

In [7]:
X_train = train[features]
y_train = train[target]

X_val = validation[features]
y_val = validation[target]

In [8]:
X_train.head()

,Vol,lag_15,lag_30,lag_60,lag_120,lag_1day,hour,day_of_week
0,216,248.0,220.0,156.0,53.0,105.0,7,4
1,260,216.0,248.0,193.0,60.0,260.0,7,4
2,264,260.0,216.0,220.0,84.0,303.0,7,4
3,69,68.0,67.0,43.0,31.0,9.0,7,4
4,221,201.0,176.0,148.0,75.0,81.0,7,4


In [9]:
X_train.dtypes

Vol              int64
lag_15         float64
lag_30         float64
lag_60         float64
lag_120        float64
lag_1day       float64
hour             int64
day_of_week      int64
dtype: object

In [10]:
X_train.isnull().sum()

Vol            0
lag_15         0
lag_30         0
lag_60         0
lag_120        0
lag_1day       0
hour           0
day_of_week    0
dtype: int64

In [11]:
y_train.isnull().sum()

np.int64(0)

In [12]:
train[["SegmentID", "Direction"]].drop_duplicates().sort_values(
    ["SegmentID", "Direction"]
)

,SegmentID,Direction
0,157501,SB
3,177615,EB
7,177615,WB
4,191114,NB


In [13]:
categorical_features = ["SegmentID", "Direction"]

numerical_features = [
    "Vol",
    "lag_15",
    "lag_30",
    "lag_60",
    "lag_120",
    "lag_1day",
    "hour",
    "day_of_week"
]

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [15]:
from sklearn.pipeline import Pipeline

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

In [17]:
model_features = numerical_features + categorical_features

X_train = train[model_features]
X_val = validation[model_features]

y_train = train[target]
y_val = validation[target]

In [18]:
X_train.columns

Index(['Vol', 'lag_15', 'lag_30', 'lag_60', 'lag_120', 'lag_1day', 'hour',
       'day_of_week', 'SegmentID', 'Direction'],
      dtype='str')

In [19]:
linear_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['Vol','lag_15','lag_30',...,'day_of_week','SegmentID','Direction']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columns

In [20]:
y_val_pred = linear_model.predict(X_val)

In [21]:
val_mae = mean_absolute_error(y_val, y_val_pred)

print("Linear Regression Validation MAE:", val_mae)

Linear Regression Validation MAE: 12.44157964357832


In [22]:
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

print("Linear Regression Validation RMSE:", val_rmse)

Linear Regression Validation RMSE: 17.20418200039816


In [23]:
persistence_val_pred = validation["Vol"]

persistence_val_mae = mean_absolute_error(
    y_val,
    persistence_val_pred
)

print("Persistence Validation MAE:", persistence_val_mae)

Persistence Validation MAE: 13.846625766871165


In [24]:
persistence_val_rmse = np.sqrt(
    mean_squared_error(y_val, persistence_val_pred)
)

print("Persistence Validation RMSE:", persistence_val_rmse)

Persistence Validation RMSE: 19.097318305515245


In [25]:
validation_results = pd.DataFrame({
    "Model": ["Persistence", "Linear Regression"],
    "MAE": [persistence_val_mae, val_mae],
    "RMSE": [persistence_val_rmse, val_rmse]
})

validation_results

,Model,MAE,RMSE
0,Persistence,13.846626,19.097318
1,Linear Regression,12.441580,17.204182
